# Notebook 3: GRPO Reinforcement Learning with Verifiable Rewards (RLVR)
**Project**: Autonomous Text-to-SQL Generator Aligned via RLVR + GRPO  
**Hardware**: Kaggle GPU (T4 x 1, P100, or 2xT4)  
**Phase**: Phase 4 (Deterministic Verifiers, Group Relative Advantages, Online Policy Optimization)

---

In [ ]:
# Step 1: Install Pinned, Compatible Dependencies for Kaggle GPU
# Pinned to match SFT Baseline (Notebook 2) and avoid TRL/Transformers ABI incompatibilities
!pip uninstall -y -q torchvision
!pip install -q \
    "transformers==4.57.3" \
    "trl==0.25.0" \
    "peft==0.18.1" \
    "datasets>=3.0.0,<5.0.0" \
    "accelerate>=1.2.1,<2.0.0" \
    "bitsandbytes>=0.45.2" \
    "sqlglot>=26.0.0"
print("Pinned training dependencies installed. Note: Session restart is recommended after fresh pip install.")

In [ ]:
# Step 2: Allocator, CUDA Config, Paths & Environment Validation
import os
# Point 9 (P1): MUST set allocator and CUDA_VISIBLE_DEVICES BEFORE importing torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
# Fully block torchvision to prevent Kaggle CUDA/PyTorch operator ABI mismatch
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
sys.modules["torchvision.ops"] = None

import json
import shutil
import zipfile
import torch
from pathlib import Path

assert torch.cuda.is_available(), "CUDA GPU is required for GRPO! Please enable GPU in Kaggle settings."
major, minor = torch.cuda.get_device_capability()
print(f"GPU: {torch.cuda.get_device_name(0)} (Compute Capability {major}.{minor})")
print(f"Initial VRAM: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
SFT_ADAPTER_DIR = WORKING_DIR / "outputs" / "sft" / "best_adapter"
GRPO_OUTPUTS_DIR = WORKING_DIR / "outputs" / "grpo"
GRPO_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Auto-unpack dataset (sql_grpo_data.zip) ───────────────────────────
train_manifest = DATA_DIR / "processed" / "train_manifest.jsonl"
dev_manifest = DATA_DIR / "processed" / "dev_manifest.jsonl"

def setup_dataset():
    if train_manifest.exists() and dev_manifest.exists():
        return
    all_zips = list(Path("/kaggle/input").glob("**/*.zip")) + list(WORKING_DIR.glob("*.zip"))
    for z in all_zips:
        if any(k in z.name.lower() for k in ["sql_grpo", "data", "processed", "spider"]):
            try:
                with zipfile.ZipFile(z, "r") as zf:
                    names = zf.namelist()
                    if any("train_manifest.jsonl" in n for n in names):
                        print(f"📦 Unpacking dataset from archive: {z} ...")
                        has_data_root = any(n.startswith("data/") for n in names)
                        extract_target = WORKING_DIR if has_data_root else DATA_DIR
                        zf.extractall(extract_target)
                        print(f"Extracted into: {extract_target}")
                        break
            except Exception as e:
                print(f"Warning inspecting {z}: {e}")
    if not train_manifest.exists() and (WORKING_DIR / "processed" / "train_manifest.jsonl").exists():
        for folder in ["processed", "raw"]:
            src = WORKING_DIR / folder
            dst = DATA_DIR / folder
            if src.exists() and not dst.exists():
                shutil.move(src, dst)
    if not train_manifest.exists():
        found_manifests = list(Path("/kaggle/input").glob("**/train_manifest.jsonl"))
        if found_manifests:
            m_path = found_manifests[0]
            dataset_root = m_path.parent.parent if m_path.parent.name == "processed" else m_path.parent
            print(f"📁 Found pre-extracted dataset at {dataset_root}. Syncing to {DATA_DIR} ...")
            src_proc = dataset_root / "processed" if (dataset_root / "processed").exists() else m_path.parent
            dst_proc = DATA_DIR / "processed"
            if src_proc.exists() and not dst_proc.exists():
                shutil.copytree(src_proc, dst_proc)
            src_raw = dataset_root / "raw" if (dataset_root / "raw").exists() else (dataset_root / "data" / "raw")
            dst_raw = DATA_DIR / "raw"
            if src_raw.exists() and not dst_raw.exists():
                shutil.copytree(src_raw, dst_raw)

setup_dataset()
assert train_manifest.exists(), "❌ train_manifest.jsonl missing! Please attach sql_grpo_data.zip in Kaggle."
with open(train_manifest, "r", encoding="utf-8") as f:
    print(f"✅ Verified dataset: {sum(1 for _ in f)} training examples available.")

# ── 2. Auto-unpack SFT Adapter (Point 5 - P0: Mandatory Adapter) ──────────
def setup_sft_adapter():
    if (SFT_ADAPTER_DIR / "adapter_model.safetensors").exists():
        return
    sft_zips = [z for z in (list(Path("/kaggle/input").glob("**/*.zip")) + list(WORKING_DIR.glob("*.zip"))) if "sft" in z.name.lower()]
    for z in sft_zips:
        try:
            with zipfile.ZipFile(z, "r") as zf:
                if any("adapter_model.safetensors" in n for n in zf.namelist()):
                    print(f"📦 Unpacking SFT adapter archive: {z} ...")
                    zf.extractall(WORKING_DIR)
                    print("SFT adapter unpacked successfully.")
                    break
        except Exception as e:
            print(f"Warning inspecting {z}: {e}")
    if not (SFT_ADAPTER_DIR / "adapter_model.safetensors").exists():
        found_adapters = (
            list(Path("/kaggle/input").glob("**/best_adapter/adapter_model.safetensors")) +
            list(Path("/kaggle/input").glob("**/adapter_model.safetensors")) +
            list(WORKING_DIR.glob("**/best_adapter/adapter_model.safetensors")) +
            list(WORKING_DIR.glob("**/adapter_model.safetensors"))
        )
        candidates = [p for p in found_adapters if p.parent != SFT_ADAPTER_DIR]
        if candidates:
            src = candidates[0].parent
            print(f"📁 Copying SFT adapter from {src} to {SFT_ADAPTER_DIR} ...")
            SFT_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
            for f in src.glob("*"):
                if f.is_file():
                    shutil.copy2(f, SFT_ADAPTER_DIR / f.name)
            print("SFT adapter copied successfully.")

setup_sft_adapter()
# Point 5 (P0): Mandate that SFT adapter exists; never silently fallback to random init
adapter_weight_file = SFT_ADAPTER_DIR / "adapter_model.safetensors"
assert adapter_weight_file.exists(), (
    f"❌ P0 ERROR: SFT Adapter weight file not found at {adapter_weight_file}! "
    f"GRPO MUST be initialized from the SFT baseline checkpoint. "
    f"Please upload and attach 'sql_grpo_sft.zip' or 'sql_grpo_sft_artifacts.zip' from Notebook 2."
)
print(f"✅ Verified SFT Adapter ready at: {SFT_ADAPTER_DIR} ({adapter_weight_file.stat().st_size / (1024**2):.2f} MB)")

In [ ]:
# Hardened Shared SQL Parser, AST Policy, Authorizer, Canonical Comparator & Sandbox
import hashlib
import json as _json
import math
import sqlite3
import time
from collections import Counter
from decimal import Decimal
from typing import Any, Dict, List, Optional, Tuple
import sqlglot
from sqlglot import exp

SAFE_FUNCTIONS = {
    "count", "sum", "avg", "min", "max", "total", "group_concat", "string_agg",
    "lower", "upper", "length", "octet_length", "substr", "substring",
    "trim", "ltrim", "rtrim", "replace", "instr", "hex", "unicode", "char", "quote",
    "round", "abs", "ceil", "ceiling", "floor", "mod", "power", "sqrt", "sign",
    "coalesce", "nullif", "ifnull", "iif",
    "strftime", "date", "time", "datetime", "julianday", "unixepoch",
    "cast", "like", "glob", "typeof",
    "rank", "dense_rank", "row_number", "ntile", "lag", "lead", "first_value", "last_value",
}
SYSTEM_TABLES = {
    "sqlite_master", "sqlite_schema", "sqlite_temp_master",
    "sqlite_temp_schema", "sqlite_sequence", "sqlite_stat1", "sqlite_stat4",
}

def extract_completion_text(c: Any) -> str:
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        for msg in reversed(c):
            if isinstance(msg, dict) and msg.get("role") == "assistant":
                return msg.get("content", "")
            if isinstance(msg, dict) and "content" in msg:
                return msg.get("content", "")
        if len(c) > 0 and isinstance(c[-1], str):
            return c[-1]
        return str(c)
    if isinstance(c, dict):
        return c.get("content", str(c))
    return str(c)

def parse_sql(text: str) -> Tuple[Optional[str], bool]:
    if not text or "<sql>" not in text or "</sql>" not in text:
        return None, False
    if text.count("<sql>") != 1 or text.count("</sql>") != 1:
        return None, False
    idx1 = text.find("<sql>") + len("<sql>")
    idx2 = text.find("</sql>")
    if idx1 >= idx2:
        return None, False
    if text[idx2 + len("</sql>"):].strip():
        return None, False
    sql = text[idx1:idx2].strip()
    if sql.startswith("```"):
        lines = sql.splitlines()[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        sql = chr(10).join(lines).strip()
    return (sql, True) if sql else (None, False)

def is_ast_safe(sql: str, valid_tables: Optional[List[str]] = None) -> bool:
    if valid_tables is not None and len(valid_tables) == 0:
        raise ValueError("is_ast_safe: valid_tables cannot be an empty list.")
    try:
        stmts = sqlglot.parse(sql, read="sqlite")
        if not stmts or len(stmts) != 1 or stmts[0] is None:
            return False
        ast = stmts[0]
        if not isinstance(ast, (exp.Select, exp.Union, exp.Intersect, exp.Except)):
            return False
        for with_node in ast.find_all(exp.With):
            if with_node.args.get("recursive"):
                return False
        for sel in ast.find_all(exp.Select):
            if not sel.expressions:
                return False
        prohibited = tuple(
            t for t in (exp.Insert, exp.Update, exp.Delete, exp.Create, exp.Drop,
                        getattr(exp, "AlterTable", getattr(exp, "Alter", None)),
                        exp.Pragma, exp.Command, exp.Transaction,
                        getattr(exp, "Attach", None), getattr(exp, "Detach", None),
                        getattr(exp, "Analyze", None)) if t is not None
        )
        if any(ast.find_all(prohibited)):
            return False
        for node in ast.walk():
            if isinstance(node, exp.Anonymous):
                if (node.name or "").lower() not in SAFE_FUNCTIONS:
                    return False
        cte_names = {c.alias.lower() for c in ast.find_all(exp.CTE) if c.alias}
        phys_tables = set()
        for t in ast.find_all(exp.Table):
            if isinstance(t.this, (exp.Anonymous, exp.Func)):
                return False
            if (t.catalog or "") or (t.db or "").lower() not in {"", "main"}:
                return False
            tn = t.name.lower() if t.name else ""
            if not tn or tn in cte_names:
                continue
            if tn in SYSTEM_TABLES or tn.startswith("sqlite_") or tn.startswith("pragma_"):
                return False
            phys_tables.add(tn)
        if valid_tables is not None:
            allowed = {str(t).lower() for t in valid_tables}
            if phys_tables - allowed:
                return False
        return True
    except Exception:
        return False

def create_authorizer(allowed_tables: Optional[List[str]] = None):
    norm_tbls = {t.lower() for t in allowed_tables} if allowed_tables is not None else None
    def callback(action, arg1, arg2, db, trigger):
        if action == sqlite3.SQLITE_SELECT:
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_READ:
            tname = (arg1 or "").lower()
            dname = (db or "").lower()
            if dname not in {"", "main"}:
                return sqlite3.SQLITE_DENY
            if tname in SYSTEM_TABLES or tname.startswith("sqlite_") or tname.startswith("pragma_"):
                return sqlite3.SQLITE_DENY
            if norm_tbls is not None and tname not in norm_tbls:
                return sqlite3.SQLITE_DENY
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_FUNCTION:
            if (arg2 or "").lower() in SAFE_FUNCTIONS:
                return sqlite3.SQLITE_OK
            return sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_DENY
    return callback

def _canonical_val(val: Any):
    if val is None:
        return ("__NULL__", None)
    if isinstance(val, bool):
        return ("bool", val)
    if isinstance(val, int):
        return ("int", val)
    if isinstance(val, float):
        if math.isnan(val):
            return ("float_nan", "nan")
        if math.isinf(val):
            return ("float_inf", 1 if val > 0 else -1)
        return ("float", 0.0 if val == 0.0 else float(val))
    if isinstance(val, Decimal):
        return ("decimal", str(val.normalize()))
    if isinstance(val, (bytes, bytearray, memoryview)):
        return ("bytes", bytes(val).hex())
    return ("str", str(val))

def canonical_row(row: Tuple[Any, ...]):
    return tuple(_canonical_val(v) for v in row)

def compare_multiset_rows(cand_rows: List[Tuple[Any, ...]], gold_rows: List[Tuple[Any, ...]]) -> bool:
    return Counter(canonical_row(r) for r in cand_rows) == Counter(canonical_row(r) for r in gold_rows)

def execute_sandbox_query(
    db_path: "Path",
    sql: str,
    timeout_sec: float = 2.0,
    row_limit: int = 5000,
    byte_limit: int = 1_048_576,
    allowed_tables: Optional[List[str]] = None,
) -> Tuple[bool, Optional[List[Tuple]], bool, float, Optional[str]]:
    start = time.perf_counter()
    deadline = start + timeout_sec
    conn = None
    cur = None
    try:
        resolved = db_path.resolve(strict=True)
        uri = resolved.as_uri() + "?mode=ro"
        conn = sqlite3.connect(uri, uri=True, timeout=1.0)
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(create_authorizer(allowed_tables=allowed_tables))
        conn.set_progress_handler(lambda: 1 if time.perf_counter() > deadline else 0, 10_000)
        cur = conn.cursor()
        cur.execute(sql)
        rows = []
        total_bytes = 0
        truncated = False
        while True:
            rem = (row_limit + 1) - len(rows)
            if rem <= 0:
                truncated = True
                break
            batch = cur.fetchmany(min(256, rem))
            if not batch:
                break
            for r in batch:
                rows.append(r)
                total_bytes += sum(
                    len(v) if isinstance(v, (bytes, bytearray))
                    else (len(v.encode("utf-8", "replace")) if isinstance(v, str) else 8)
                    for v in r
                )
                if len(rows) > row_limit or total_bytes > byte_limit:
                    truncated = True
                    break
            if truncated:
                break
        dur_ms = (time.perf_counter() - start) * 1000.0
        if truncated:
            return False, None, True, dur_ms, "RESULT_TRUNCATED"
        return True, rows, False, dur_ms, None
    except Exception as e:
        dur_ms = (time.perf_counter() - start) * 1000.0
        return False, None, False, dur_ms, str(e)
    finally:
        if cur:
            try:
                cur.close()
            except Exception:
                pass
        if conn:
            try:
                conn.set_progress_handler(None, 0)
                conn.set_authorizer(None)
                conn.close()
            except Exception:
                pass


def reward_format(completions, **kwargs):
    rewards = []
    for c in completions:
        _, ok = parse_sql(extract_completion_text(c))
        rewards.append(0.20 if ok else -0.50)
    return rewards

def reward_ast_schema(completions, valid_tables=None, **kwargs):
    rewards = []
    for i, c in enumerate(completions):
        sql, ok = parse_sql(extract_completion_text(c))
        if not ok or not sql:
            rewards.append(0.00)
            continue
        tbls = list(valid_tables[i]) if (valid_tables is not None and i < len(valid_tables) and valid_tables[i] is not None) else None
        try:
            rewards.append(0.20 if is_ast_safe(sql, tbls) else 0.00)
        except Exception:
            rewards.append(0.00)
    return rewards

def reward_exec_valid(completions, db_id=None, valid_tables=None, **kwargs):
    rewards = []
    for i, c in enumerate(completions):
        sql, ok = parse_sql(extract_completion_text(c))
        target_db = db_id[i] if (db_id and i < len(db_id)) else ""
        tbls = list(valid_tables[i]) if (valid_tables is not None and i < len(valid_tables) and valid_tables[i] is not None) else None
        if not ok or not sql or not target_db:
            rewards.append(-0.20)
            continue
        try:
            if not is_ast_safe(sql, tbls):
                rewards.append(-0.20)
                continue
        except Exception:
            rewards.append(-0.20)
            continue
        db_file = DATA_DIR / "raw" / "spider" / "database" / target_db / f"{target_db}.sqlite"
        if not db_file.exists():
            rewards.append(-0.20)
            continue
        success, _, trunc, _, _ = execute_sandbox_query(db_file, sql, timeout_sec=2.0, allowed_tables=tbls)
        rewards.append(0.30 if (success and not trunc) else -0.20)
    return rewards

def reward_accuracy(completions, db_id=None, gold_result_json=None, gold_is_empty=None, empty_gold_verified=None, valid_tables=None, **kwargs):
    rewards = []
    for i, c in enumerate(completions):
        sql, ok = parse_sql(extract_completion_text(c))
        target_db = db_id[i] if (db_id and i < len(db_id)) else ""
        tbls = list(valid_tables[i]) if (valid_tables is not None and i < len(valid_tables) and valid_tables[i] is not None) else None
        if not ok or not sql or not target_db or not gold_result_json or i >= len(gold_result_json):
            rewards.append(0.00)
            continue
        try:
            if not is_ast_safe(sql, tbls):
                rewards.append(0.00)
                continue
        except Exception:
            rewards.append(0.00)
            continue
        db_file = DATA_DIR / "raw" / "spider" / "database" / target_db / f"{target_db}.sqlite"
        if not db_file.exists():
            rewards.append(0.00)
            continue
        success, cand_rows, trunc, _, _ = execute_sandbox_query(db_file, sql, timeout_sec=2.0, allowed_tables=tbls)
        if not success or trunc or cand_rows is None:
            rewards.append(0.00)
            continue
        try:
            gold_raw = json.loads(gold_result_json[i])
            gold_rows = [tuple(x) if isinstance(x, list) else (x,) for x in gold_raw]
            is_empty = bool(gold_is_empty[i]) if (gold_is_empty and i < len(gold_is_empty)) else (len(gold_rows) == 0)
            if is_empty:
                # Empty-gold anti-hacking: never reward empty agreement unless independently verified
                is_ver = bool(empty_gold_verified[i]) if (empty_gold_verified and i < len(empty_gold_verified)) else False
                rewards.append(1.00 if (is_ver and len(cand_rows) == 0) else 0.00)
            else:
                rewards.append(1.00 if compare_multiset_rows(cand_rows, gold_rows) else 0.00)
        except Exception:
            rewards.append(0.00)
    return rewards

print("✅ 4 Hardened Verifiable Reward Functions loaded (AST + authorizer + byte cap + empty-gold guard).")

In [ ]:
# Step 3b: Reward Function Unit Tests (Point 11 - P1)
print("Running comprehensive unit tests for all reward functions...")

# Find any available sqlite database in the dataset for live execution test
sample_dbs = list((DATA_DIR / "raw" / "spider" / "database").glob("*/*.sqlite"))
assert len(sample_dbs) > 0, "No SQLite databases found for testing!"
test_db_id = sample_dbs[0].parent.name
test_db_path = sample_dbs[0]

# Inspect a table in test db
with sqlite3.connect(test_db_path) as c:
    tbls = [r[0] for r in c.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
test_table = tbls[0] if tbls else "test"

# 1. Test conversational completion extraction
str_comp = f"<think>reasoning</think><sql>SELECT * FROM {test_table} LIMIT 1</sql>"
dict_comp = {"role": "assistant", "content": f"<think>ok</think><sql>SELECT * FROM {test_table} LIMIT 1</sql>"}
list_comp = [{"role": "user", "content": "query"}, {"role": "assistant", "content": f"<think>ok</think><sql>SELECT * FROM {test_table} LIMIT 1</sql>"}]
bad_comp = "Just some text without tags"

assert extract_completion_text(str_comp) == str_comp
assert "<sql>" in extract_completion_text(dict_comp)
assert "<sql>" in extract_completion_text(list_comp)

# 2. Test reward_format
fmt_res = reward_format([str_comp, dict_comp, list_comp, bad_comp])
assert fmt_res == [0.20, 0.20, 0.20, -0.50], f"reward_format test failed: {fmt_res}"

# 3. Test reward_ast_schema
ast_res = reward_ast_schema(
    [str_comp, f"<sql>SELECT * FROM fake_nonexistent_table_xyz</sql>"],
    valid_tables=[[test_table], [test_table]]
)
assert ast_res[0] == 0.20 and ast_res[1] == 0.00, f"reward_ast_schema test failed: {ast_res}"

# 4. Test reward_exec_valid
exec_res = reward_exec_valid(
    [str_comp, f"<sql>SELECT invalid_syntax FROM</sql>"],
    db_id=[test_db_id, test_db_id]
)
assert exec_res[0] == 0.30 and exec_res[1] == -0.20, f"reward_exec_valid test failed: {exec_res}"

# 5. Test reward_accuracy
with sqlite3.connect(test_db_path) as c:
    gold_rows = [list(r) for r in c.execute(f"SELECT * FROM {test_table} LIMIT 1").fetchall()]
gold_json = json.dumps(gold_rows)

acc_res = reward_accuracy(
    [str_comp, f"<sql>SELECT * FROM {test_table} WHERE 1=0</sql>"],
    db_id=[test_db_id, test_db_id],
    gold_result_json=[gold_json, gold_json],
    gold_is_empty=[False, False]
)
assert acc_res[0] == 1.00 and acc_res[1] == 0.00, f"reward_accuracy test failed: {acc_res}"

print("🎉 ALL REWARD FUNCTION UNIT TESTS PASSED SUCCESSFULLY!")
# 6. Anti-hacking empty-gold & pragma TVF bypass tests
assert reward_ast_schema(["<sql>SELECT * FROM pragma_table_info('x')</sql>"], valid_tables=[[test_table]])[0] == 0.00
assert reward_accuracy([f"<sql>SELECT * FROM {test_table} WHERE 1=0</sql>"], db_id=[test_db_id], gold_result_json=["[]"], gold_is_empty=[True], valid_tables=[[test_table]])[0] == 0.00
print("✅ Empty-gold anti-hacking & pragma TVF tests passed!")

In [ ]:
# Step 4: Load Base Model, Set Left-Padding & Resume Mandatory SFT Adapter
import sys
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
sys.modules["torchvision.ops"] = None

import transformers.utils.import_utils as _imp
_imp.is_torchvision_available = lambda: False

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Point 6 (P1): Decoder-only generation requires left padding
tokenizer.padding_side = "left"
print(f"Tokenizer configured with padding_side='{tokenizer.padding_side}' and pad_token_id={tokenizer.pad_token_id}")

major, minor = torch.cuda.get_device_capability()
supports_bf16 = (major >= 8)
# On T4 (CC 7.5), native compute dtype is float16
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
print(f"CUDA Capability: {major}.{minor} | Using 4-bit compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading 4-bit base model: {MODEL_ID} ...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    trust_remote_code=True,
)

# Point 5 (P0): Mandate that SFT adapter exists; resume adapter strictly
adapter_weight_path = SFT_ADAPTER_DIR / "adapter_model.safetensors"
assert adapter_weight_path.exists(), f"❌ Mandatory SFT adapter weights missing at {adapter_weight_path}!"
print(f"Resuming policy from mandatory SFT Adapter: {SFT_ADAPTER_DIR} ...")
model = PeftModel.from_pretrained(base_model, str(SFT_ADAPTER_DIR), is_trainable=True)

# Clean any rogue bfloat16 tensors to prevent GradScaler mismatch on T4
for p in model.parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)
for b in model.buffers():
    if b.dtype == torch.bfloat16:
        b.data = b.data.to(torch.float16)

# Prevent HuggingFace Trainer from wrapping 4-bit model in DataParallel
model.is_parallelizable = True
model.model_parallel = True
if hasattr(model, "base_model"):
    model.base_model.is_parallelizable = True
    model.base_model.model_parallel = True

model.print_trainable_parameters()
print(f"Model ready on {model.device}. Peak VRAM: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

In [ ]:
# Step 5: Dataset Validation & Reproducible Sampling (Point 10 - P1)
import random
from datasets import Dataset

train_manifest = DATA_DIR / "processed" / "train_manifest.jsonl"
assert train_manifest.exists(), f"train_manifest.jsonl not found at {train_manifest}!"

raw_records = []
with open(train_manifest, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_records.append(json.loads(line))

print(f"Loaded {len(raw_records)} total records from manifest.")

# Validate records and verify SQLite databases exist
valid_records = []
for r in raw_records:
    db_file = DATA_DIR / "raw" / "spider" / "database" / r["db_id"] / f"{r['db_id']}.sqlite"
    if not db_file.exists():
        continue
    valid_records.append({
        "prompt": r["prompt"],
        "db_id": r["db_id"],
        "valid_tables": r["valid_tables"],
        "gold_result_json": r["gold_result_json"],
        "gold_is_empty": r.get("gold_is_empty", False),
    })

print(f"Verified {len(valid_records)} records with intact SQLite databases.")
assert len(valid_records) >= 5000, f"Expected >=5000 valid records, found {len(valid_records)}"

# Reproducible sampling with seed=42 across all databases
random.seed(42)
TRAIN_SAMPLE_SIZE = min(500, len(valid_records))  # 500 diverse prompts for pilot GRPO
sampled_records = random.sample(valid_records, TRAIN_SAMPLE_SIZE)
grpo_dataset = Dataset.from_list(sampled_records)

unique_dbs = len({r['db_id'] for r in sampled_records})
print(f"✅ Selected {len(grpo_dataset)} reproducible GRPO prompts spanning {unique_dbs} distinct databases.")

In [ ]:
# Step 6: Configure GRPOTrainer with Group Size G=8 and Batch Divisibility
import inspect
import torch
from trl import GRPOTrainer, GRPOConfig

NUM_GENERATIONS = 8 # G=8 candidate queries per prompt
major, minor = torch.cuda.get_device_capability()
use_bf16 = (major >= 8)
use_fp16 = True # Native FP16 on T4
print(f"CUDA Capability: {major}.{minor} | Training precision: bf16={use_bf16}, fp16={use_fp16} | Device: {torch.cuda.get_device_name(0)}")

raw_grpo_args = {
    "output_dir": str(GRPO_OUTPUTS_DIR),
    "num_generations": NUM_GENERATIONS,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8, # 1 * 8 = 8, divisible by num_generations=8
    "temperature": 0.8,
    "learning_rate": 5.0e-6,
    "beta": 0.04, # KL regularization
    "max_prompt_length": 1024,
    "max_completion_length": 256,
    "max_steps": 100,
    "logging_steps": 5,
    "save_steps": 25,
    "fp16": use_fp16,
    "bf16": use_bf16,
    "remove_unused_columns": False, # Essential: preserves custom metadata for reward callables
    "report_to": "none",
}

grpo_sig = inspect.signature(GRPOConfig.__init__).parameters
has_var_kw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in grpo_sig.values())
valid_grpo_args = raw_grpo_args if has_var_kw else {k: v for k, v in raw_grpo_args.items() if k in grpo_sig}
grpo_args = GRPOConfig(**valid_grpo_args)

# Force single GPU mode inside TrainingArguments so Trainer NEVER invokes DataParallel
grpo_args._n_gpu = 1

reward_funcs = [
    reward_format,
    reward_ast_schema,
    reward_exec_valid,
    reward_accuracy,
]

trainer_kwargs = {
    "model": model,
    "reward_funcs": reward_funcs,
    "args": grpo_args,
    "train_dataset": grpo_dataset,
}
trainer_sig = inspect.signature(GRPOTrainer.__init__).parameters
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = GRPOTrainer(**trainer_kwargs)

print("🚀 GRPOTrainer initialized. Starting RLVR Fine-Tuning...")
trainer.train()

# Save Final GRPO Policy Adapter
final_adapter_dir = GRPO_OUTPUTS_DIR / "final_adapter"
trainer.model.save_pretrained(str(final_adapter_dir))
tokenizer.save_pretrained(str(final_adapter_dir))
print(f"✅ Successfully saved Final GRPO Adapter to: {final_adapter_dir}")

In [ ]:
# Step 7: Held-Out Evaluation on Spider Dev Set (Hardened AST + Sandbox + Canonical Multiset)
print("Running Held-Out Evaluation on Spider Dev Set...")
EVAL_MODE = "first_100"  # Change to "full" for all 1,034 dev examples
dev_manifest = DATA_DIR / "processed" / "dev_manifest.jsonl"
assert dev_manifest.exists(), f"dev_manifest.jsonl missing at {dev_manifest}!"
eval_samples = [json.loads(line) for line in open(dev_manifest, "r", encoding="utf-8") if line.strip()]
eval_subset = eval_samples if EVAL_MODE == "full" else eval_samples[:min(100, len(eval_samples))]
print(f"Evaluating {len(eval_subset)} held-out dev examples (EVAL_MODE={EVAL_MODE})...")

model.eval()
valid_format_cnt = ast_safe_cnt = valid_exec_cnt = exec_correct_cnt = 0
for i, item in enumerate(eval_subset):
    prompt_text = tokenizer.apply_chat_template(item["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=None, pad_token_id=tokenizer.pad_token_id)
    generated = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    sql, ok = parse_sql(generated)
    if ok and sql:
        valid_format_cnt += 1
        v_tbls = item.get("valid_tables")
        if is_ast_safe(sql, v_tbls if v_tbls else None):
            ast_safe_cnt += 1
            db_id = item["db_id"]
            db_path = DATA_DIR / "raw" / "spider" / "database" / db_id / f"{db_id}.sqlite"
            if db_path.exists():
                ok_exec, rows, trunc, _, _ = execute_sandbox_query(db_path, sql, timeout_sec=2.0, allowed_tables=v_tbls)
                if ok_exec and not trunc and rows is not None:
                    valid_exec_cnt += 1
                    gold_rows = [tuple(x) if isinstance(x, list) else (x,) for x in json.loads(item["gold_result_json"])]
                    if compare_multiset_rows(rows, gold_rows):
                        exec_correct_cnt += 1
    if (i + 1) % 25 == 0:
        print(f"  [{i+1}/{len(eval_subset)}] Current EX%: {exec_correct_cnt / (i + 1) * 100:.1f}%")

grpo_metrics = {
    "eval_mode": EVAL_MODE,
    "eval_size": len(eval_subset),
    "format_validity_pct": round(valid_format_cnt / len(eval_subset) * 100, 2),
    "ast_safety_pct": round(ast_safe_cnt / len(eval_subset) * 100, 2),
    "execution_validity_pct": round(valid_exec_cnt / len(eval_subset) * 100, 2),
    "execution_accuracy_ex_pct": round(exec_correct_cnt / len(eval_subset) * 100, 2),
}
print("\n🎯 Final GRPO Policy Held-Out Evaluation Results:")
print(json.dumps(grpo_metrics, indent=2))
metrics_file = GRPO_OUTPUTS_DIR / "grpo_evaluation_metrics.json"
metrics_file.write_text(json.dumps(grpo_metrics, indent=2), encoding="utf-8")
print(f"Saved metrics to: {metrics_file}")

In [ ]:
# ==========================================================================
# FINAL CELL: MANDATORY ARTIFACT ZIPPER
# Zips the trained GRPO policy adapter, evaluation logs, and metrics for download
# ==========================================================================
import zipfile

output_zip_path = WORKING_DIR / "sql_grpo_rlvr_artifacts.zip"
print(f"Creating artifact zip at: {output_zip_path} ...")

with zipfile.ZipFile(output_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(GRPO_OUTPUTS_DIR):
        for file in files:
            p = Path(root) / file
            # Store with forward slash paths for Linux & Windows cross-compatibility
            rel = p.relative_to(WORKING_DIR).as_posix()
            zipf.write(p, arcname=rel)

zip_size_mb = os.path.getsize(output_zip_path) / (1024 * 1024)
print(f"✅ Successfully created: {output_zip_path.name} ({zip_size_mb:.2f} MB)")
from IPython.display import FileLink, display
print("\n👉 Click the link below to download directly in your browser:")
display(FileLink("sql_grpo_rlvr_artifacts.zip"))